# 03 - Build Daily Generation Fact Table

## Overview

This notebook estimates daily electricity generation for the renewable asset portfolio prepared in the previous notebooks.

The Renewables-Climate Mart contains registered generation assets and their installed capacities, but not measured asset-level generation values. Therefore, this notebook derives daily generation estimates by combining portfolio-level annual generation targets with technology-specific generation profiles and the available asset capacity over time.

The estimation process uses annual renewable generation totals as target values and distributes them across days, energy sources, TSO regions, and individual assets. SMARD generation profiles are used to represent the temporal distribution of generation, while commissioning dates ensure that assets only contribute after becoming operational.

The resulting fact table (`fact_generation_daily`) provides daily asset-level generation estimates and is used as the central generation fact table of the Renewables-Climate Mart.

## Input Data

This notebook uses the processed dimension tables created in Notebook 02:

- `data/processed/dim_asset.csv`
- `data/processed/dim_area_h.csv`
- `data/processed/dim_company_h.csv`
- `data/processed/dim_energy_source_h.csv`
- `data/processed/dim_date.csv`

In addition, daily generation profiles from SMARD are used as external reference data.

## Output

The notebook exports the estimated generation fact table to:

- `data/processed/fact_generation_daily.csv`

In [1]:
import pandas as pd

import numpy as np
import itertools

In [2]:
DATA_DIR = "../data"

RAW_SMARD_DIR = f"{DATA_DIR}/raw/smard"
PROCESSED_DIR = f"{DATA_DIR}/processed"

## 1. Load Dimension Tables and Define Generation Targets

This section loads the processed dimension tables created in the previous notebook and defines the annual generation targets used for the estimation process.

The total renewable generation and biomass generation values are taken from the respective annual reports. Hydro generation is added as a fixed external estimate. Since measured asset-level generation values are not available, the remaining generation volume is assigned to wind and solar and distributed in later steps.

These target values provide the annual totals that the daily generation estimates are calibrated against.

In [3]:
dim_company_h = pd.read_csv(f"{PROCESSED_DIR}/dim_company_h.csv")
dim_area_h = pd.read_csv(f"{PROCESSED_DIR}/dim_area_h.csv")
dim_energy_source_h = pd.read_csv(f"{PROCESSED_DIR}/dim_energy_source_h.csv")
dim_asset = pd.read_csv(f"{PROCESSED_DIR}/dim_asset.csv")
dim_date = pd.read_csv(f"{PROCESSED_DIR}/dim_date.csv")

In [4]:
# Taken from the respective annual reports
enercity_renewable_2024 = 1968000
enercity_renewable_2025 = 2305000

enercity_biomass_2024 = 647000
enercity_biomass_2025 = 972000

# Taken from various sources online
enercity_hydro_2025 = enercity_hydro_2024 = 6400

enercity_windAndSolar_2024 = enercity_renewable_2024 - enercity_biomass_2024 - enercity_hydro_2024
enercity_windAndSolar_2025 = enercity_renewable_2025 - enercity_biomass_2025 - enercity_hydro_2025

## 2. Estimate Wind and Solar Generation Targets

This section estimates how the combined wind and solar generation volume is split between the two technologies.

Since the available annual reporting data does not provide separate generation values for wind and solar, the split is approximated using MW-days as a capacity-weighted availability measure. For 2024, the MW-day calculation results in a solar share of approximately 2.66%. However, this value is adjusted downward to 2.0% because installed solar capacity cannot be translated linearly into generation: photovoltaic generation has a lower effective capacity factor than wind power and is limited to daylight hours with strong seasonal variation.

For 2025, the wind and solar shares are updated using changes in MW-days and simple weather-related adjustment factors. The resulting shares are then applied to the combined wind and solar generation target.

In [5]:
def calculate_mwd(df_assets, year):
    # Define start and end date of the year
    start_of_year = pd.to_datetime(f'{year}-01-01')
    end_of_year = pd.to_datetime(f'{year}-12-31')
    
    # Making sure commissioning_date is a datetime-object
    commissioned = pd.to_datetime(df_assets['commissioning_date'])

    # Set commissioning_dates preceding the year to the beginning of the year
    active_start = commissioned.clip(lower=start_of_year)
    
    # Calculate the active days within the year
    delta = (end_of_year - active_start).dt.days + 1
    
    # Set negative active days to 0 (for commissioning_dates succeeding the end of the year)
    days_active = delta.clip(lower=0)

    # MWd = capacity * active days
    mwd = (df_assets['generation_capacity_MW'] * days_active).sum()
    return mwd


wind_asset = dim_asset[dim_asset['energy_source_id']==2].merge(dim_date, left_on='commissioning_date_id', right_on='date_id', how='left').rename(columns={'date':'commissioning_date'})
solar_asset = dim_asset[dim_asset['energy_source_id']==4].merge(dim_date, left_on='commissioning_date_id', right_on='date_id', how='left').rename(columns={'date':'commissioning_date'})

mwd24_wind = calculate_mwd(wind_asset,2024)
mwd24_solar = calculate_mwd(solar_asset,2024)

shares = pd.Series({
    "Wind": mwd24_wind,
    "Solar": mwd24_solar
})

shares = shares / shares.sum()
shares



Wind     0.973373
Solar    0.026627
dtype: float64

In [6]:
# The MW-day based solar share is adjusted downward to 2% to account for lower PV utilization compared with wind generation.

enercity_wind_2024 = round(enercity_windAndSolar_2024 * 0.98, -2)
enercity_solar_2024 = round(enercity_windAndSolar_2024 * 0.02, -2)

print(enercity_wind_2024)
print(enercity_solar_2024)

1288300.0
26300.0


In [7]:
# 2025 split: update the 2024 split using MW-day growth and weather-related adjustment factors.

mwd25_wind = calculate_mwd(wind_asset,2025)
mwd25_solar = calculate_mwd(solar_asset,2025)

growth_factor_wind = mwd25_wind/mwd24_wind
growth_factor_solar = mwd25_solar/mwd24_solar

# Weather adjustment factors between 2024 and 2025
# Wind: Fraunhofer ISE reported 3.2% lower wind generation due to weaker wind conditions.
# Solar: Based on DWD annual weather reports, sunshine duration increased from 1,700 h (2024) to 1,945 h (2025), corresponding to a factor of 1.144.

expect_25_wind = enercity_wind_2024 * growth_factor_wind * 0.968
expect_25_solar = enercity_solar_2024 * growth_factor_solar * 1.144

total = expect_25_solar + expect_25_wind
share_wind = expect_25_wind / total
share_solar = expect_25_solar /  total

print(share_wind)
print(share_solar)

enercity_wind_2025 = round(enercity_windAndSolar_2025 * share_wind, -2)
enercity_solar_2025 = round(enercity_windAndSolar_2025 * share_solar, -2)

print(enercity_wind_2025)
print(enercity_solar_2025)

0.9586092457984746
0.04139075420152534
1271700.0
54900.0


## 3. Define Generation Allocation Functions

This section defines the helper functions used to distribute annual generation targets to daily asset-level records.

The SMARD generation profiles provide the daily temporal distribution for each energy source and TSO region. For each year, energy source, and TSO region, the model calculates the available portfolio capacity over time and derives a daily generation value per available MW.

The allocation is then expanded to individual assets. Daily generation is distributed across assets based on installed capacity with a random variation factor to avoid unrealistically flat asset-level profiles.

Commissioning dates are respected so that assets only produce electricity after they become operational. The resulting output contains daily generation estimates by `date_id` and `asset_id`.

In [8]:
def get_smard_attributes(energy_source):
    if energy_source == 'Wind':
        return 'Wind onshore [MWh] Calculated resolutions'
    if energy_source == 'Biomass':
        return 'Biomass [MWh] Calculated resolutions'
    if energy_source == 'Hydro':
        return 'Hydropower [MWh] Calculated resolutions'
    if energy_source == 'Solar':
        return 'Photovoltaics [MWh] Calculated resolutions'
    else:
        return

# Filter assets commissioned ON or BEFORE this specific day    
def get_capacity_at_time(current_time, sourceAsset_df):
    is_live = sourceAsset_df['commissioning_date'] <= current_time
    return sourceAsset_df.loc[is_live, 'generation_capacity_MW'].sum()

def get_capacity_of_TSO(current_time, sourceAsset_df, tso):
    is_live = (sourceAsset_df['commissioning_date'] <= current_time) & (sourceAsset_df['tso_region'] == tso)
    return sourceAsset_df.loc[is_live, 'generation_capacity_MW'].sum()

# Retrieve the annual target variable for the selected year and energy source.
def get_total_generation(year, energy_source):
    return eval('enercity_' + energy_source.lower() + '_' + str(year))

In [9]:
def generate_calc_df_for_source_and_region(year, energy_source, tso_region):
    column = get_smard_attributes(energy_source)
    tso_df = pd.read_csv(f"{RAW_SMARD_DIR}/Actual_generation_" + str(year) + "_Day_" + tso_region + ".csv",sep=None, engine='python')
    enercity_total_gen = get_total_generation(year,energy_source)
    
    tso_df[column] = tso_df[column].str.replace(',','').astype(float) if tso_df[column].dtype == 'object' else tso_df[column]
    tso_source_year = tso_df[column].sum()

    source_calc = pd.DataFrame()

    # Build the asset "universe" for the selected energy source, including geographic information and commissioning dates required for capacity calculations.
    merge_df = pd.merge(dim_asset, dim_area_h, on ='area_id', how='left').merge(dim_energy_source_h, on= 'energy_source_id', how='left').merge(dim_date, left_on='commissioning_date_id', right_on='date_id', how='left').rename(columns={'date':'commissioning_date'})
    sourceAsset_df = merge_df[merge_df.energy_source_name==energy_source].copy()

    source_calc['timestamp'] = dim_date[dim_date['year']==year]['date'].reset_index(drop=True)
    source_calc['tso_generation'] = tso_df[column]

    # Calculate each day’s share of the annual SMARD generation profile.
    source_calc['day_share'] = source_calc['tso_generation'] / tso_source_year

    # Calculate the commissioned portfolio capacity available on each day, both across the full portfolio and within the selected TSO region.
    source_calc['available_capacity'] = source_calc.apply(lambda x:get_capacity_at_time(x['timestamp'],sourceAsset_df), axis=1)
    source_calc['available_capacity_tso'] = source_calc.apply(lambda x:get_capacity_of_TSO(x['timestamp'],sourceAsset_df, tso_region), axis=1)

    # Allocate the annual portfolio target to the TSO region based on available capacity.
    source_calc['capacity_share_tso'] = np.where(
    source_calc['available_capacity'] > 0,
    source_calc['available_capacity_tso'] / source_calc['available_capacity'],
    0
    )

    # Compare the theoretical generation implied by installed capacity with the generation target allocated to the portfolio and TSO region.
    source_calc['theoretical_generation'] = source_calc['available_capacity_tso'] * source_calc['day_share']
    source_calc['enercity_total_gen_by_source_and_tso'] = source_calc['day_share'] * enercity_total_gen * source_calc['capacity_share_tso']
    
    total_target = source_calc['enercity_total_gen_by_source_and_tso'].sum()
    total_ideal = source_calc['theoretical_generation'].sum()

    if total_ideal <= 0 or pd.isna(total_ideal):
        source_calc['gen_by_each_MW'] = 0
        return source_calc, sourceAsset_df

    # Derive a scaling factor that converts the theoretical generation profile into the desired annual generation target.
    scaling_factor = total_target / total_ideal
    source_calc['gen_by_each_MW'] = scaling_factor * source_calc['day_share']

    return source_calc, sourceAsset_df

In [10]:
def calc_fact_for_source_and_region(source_calc, sourceAsset_df, tso_region):
    
    # 1. Filter for only assets from a specific tso_region
    sourceAssetTso_df = sourceAsset_df[sourceAsset_df['tso_region']==tso_region].copy()

    # 2. Add a temporary key to both DataFrames to allow a cross-merge and perform the cross-merge
    sourceAssetTso_df['key'] = 1
    source_calc['key'] = 1
    fact = pd.merge(source_calc, sourceAssetTso_df, on='key').drop('key', axis=1)


    # -------------------------------------------------------------------------
    # 3. RANDOMIZED CALCULATION
    # -------------------------------------------------------------------------
    
    # Calculate the base generation to reach daily targets
    fact['base_generation_MWh'] = fact['generation_capacity_MW'] * fact['gen_by_each_MW']
    
    # Get the exact sum for each day in this TSO
    fact['daily_target_MWh'] = fact.groupby('timestamp')['base_generation_MWh'].transform('sum')

    # Generate random asset-level weights (Normal distribution centered at 1.0, with 30% standard deviation)
    # Clip the lower end to avoid negative weights
    fact['noise_factor'] = np.random.normal(loc=1.0, scale=0.30, size=len(fact))
    fact['noise_factor'] = fact['noise_factor'].clip(lower=0.01)
    
    # Calculate the "claim" each plant has on the day's energy
    fact['noisy_weight'] = fact['generation_capacity_MW'] * fact['noise_factor']
    
    # Normalize the weights per day and allocate the energy
    fact['daily_weight_sum'] = fact.groupby('timestamp')['noisy_weight'].transform('sum')
    fact['generation_MWh'] = fact['daily_target_MWh'] * (fact['noisy_weight'] / fact['daily_weight_sum'])

    # Clean up calculation columns    
    fact = fact.drop(columns=['base_generation_MWh', 'daily_target_MWh', 'noise_factor', 'noisy_weight', 'daily_weight_sum'])
    
    # -------------------------------------------------------------------------

    # 4. Zero out generation if the day is BEFORE the commissioning date
    fact['timestamp'] = pd.to_datetime(fact['timestamp'])
    fact['commissioning_date'] = pd.to_datetime(fact['commissioning_date'])

    fact.loc[fact['timestamp'] < fact['commissioning_date'], 'generation_MWh'] = 0

    # 5. Create the market_day_id
    fact['date_id'] = fact['timestamp'].dt.strftime('%Y%m%d').astype(int)

    # 6. Final selection
    fact_generation_source_tso = fact[['date_id', 'asset_id', 'generation_MWh']]

    return fact_generation_source_tso

## 4. Generate Daily Asset-Level Generation Fact Table

This section applies the allocation functions to generate the daily generation fact table.

The model iterates over all combinations of year, energy source, and TSO region. For each combination, daily generation profiles are calculated and expanded to asset-level records. To avoid identical generation profiles for all assets of the same energy source, the allocation incorporates randomized asset-level weights. A fixed random seed ensures that the resulting estimates remain reproducible across notebook executions. The four TSO-specific results are then combined for each year and energy source.

After the initial allocation, a global correction factor is applied so that the generated values match the annual generation target for the respective year and energy source. A final capacity check ensures that daily generation does not exceed the physical production limit of each asset. Remaining zero-generation records are removed from the final fact table.

In [11]:
np.random.seed(42)

years = [2024, 2025]
sources = ['Wind', 'Solar', 'Biomass', 'Hydro']
tsos = ['TenneT', '50Hertz', 'Amprion', 'TransnetBW']

all_fact_dfs = []

# 1. Loop only over years and energy_sources
for year, source in itertools.product(years, sources):
    
    source_year_dfs = []
    
    # 2. Calculate for each tso the given year and energy_source
    for tso in tsos:
        source_calc, sourceAsset_df = generate_calc_df_for_source_and_region(year, source, tso)
        fact_gen = calc_fact_for_source_and_region(source_calc, sourceAsset_df, tso)
        source_year_dfs.append(fact_gen)
        
    # 3. Concat the data for all 4 tso_regions
    combined_source_year = pd.concat(source_year_dfs, ignore_index=True)

    # 4. Scale the generated values to match the annual target for this year and source.
    current_total = combined_source_year['generation_MWh'].sum()
    target_total = get_total_generation(year, source) 
    
    if current_total > 0:
        correction_factor = target_total / current_total
        combined_source_year['generation_MWh'] *= correction_factor

        # Apply physical daily capacity limit after annual scaling.
        combined_source_year = pd.merge(combined_source_year, dim_asset[['asset_id', 'generation_capacity_MW']], on='asset_id', how='left')
        max_hours = 15.95 if source == "Solar" else 23.95
        combined_source_year['max_daily_MWh'] = combined_source_year['generation_capacity_MW'] * max_hours
        
        # Loop to ensure redistribution doesn't push assets over the limit
        for _ in range(5): # Usually 2-3 iterations are enough
            overflow_mask = combined_source_year['generation_MWh'] > combined_source_year['max_daily_MWh']
            if not overflow_mask.any():
                break
                
            total_overflow = (combined_source_year.loc[overflow_mask, 'generation_MWh'] - combined_source_year['max_daily_MWh']).sum()
            combined_source_year.loc[overflow_mask, 'generation_MWh'] = combined_source_year['max_daily_MWh']
            
            under_mask = combined_source_year['generation_MWh'] < combined_source_year['max_daily_MWh']
            if under_mask.any() and total_overflow > 0:
                # Distribute ONLY to the available capacity (the 'room')
                room_available = (combined_source_year['max_daily_MWh'] - combined_source_year['generation_MWh']).clip(lower=0)
                total_room = room_available[under_mask].sum()
                
                if total_room > total_overflow:
                    # Distribute proportionally to existing generation
                    dist_factor = total_overflow / combined_source_year.loc[under_mask, 'generation_MWh'].sum()
                    combined_source_year.loc[under_mask, 'generation_MWh'] *= (1 + dist_factor)
                else:
                    # Fill each asset to the max.
                    combined_source_year.loc[under_mask, 'generation_MWh'] = combined_source_year['max_daily_MWh']
                    # Any remaining total_overflow stays 'lost' to maintain physical reality.
        
        # Cleanup temporary columns
        combined_source_year = combined_source_year.drop(columns=['generation_capacity_MW', 'max_daily_MWh'])

    # 5. Append the corrected subset to the master-list
    all_fact_dfs.append(combined_source_year)

# 6. Combine all subsets into the final fact table
final_fact_df = pd.concat(all_fact_dfs, ignore_index=True)

final_fact_df = final_fact_df[final_fact_df['generation_MWh']!=0].reset_index(drop=True)

## 5. Validate and Export Generation Fact Table

This section performs basic validation checks on the generated fact table before exporting it.

The generated values are joined with the dimension tables to inspect annual totals by year and energy source and to check whether individual daily generation values remain within plausible utilization limits. Solar assets are evaluated using a lower maximum daily usage assumption than the other technologies to reflect daylight-hour constraints.

After validation, the final fact table is exported to `data/processed/fact_generation_daily.csv`.

In [12]:
merge_df = pd.merge(final_fact_df, dim_asset, on = 'asset_id', how='left').merge(dim_area_h, on='area_id', how='left').merge( dim_date, on = 'date_id', how='left').merge(dim_company_h, on = 'company_id', how='left').merge(dim_energy_source_h, on='energy_source_id', how='left')

# 1. Find the row with the maximum generation for each Year and Source
idx = merge_df.groupby(['year', 'energy_source_name'])['generation_MWh'].idxmax()
max_gen_df = merge_df.loc[idx]

# 2. Iterate through these max rows
for _, row in max_gen_df.iterrows():
    energy_source = row['energy_source_name']
    year = row['year']
    
    # Use a lower maximum daily production window for solar assets.
    max_usage = 16 if energy_source == 'Solar' else 24
    
    # Calculate utilization
    utilization = row['generation_MWh'] / (row['generation_capacity_MW'] * max_usage)
    
    print(f"{energy_source} - {year}")
    print(f"Max utilization: {utilization:.4f}")
    print('---------------------------------')

Biomass - 2024
Max utilization: 0.8640
---------------------------------
Hydro - 2024
Max utilization: 0.9141
---------------------------------
Solar - 2024
Max utilization: 0.6612
---------------------------------
Wind - 2024
Max utilization: 0.9841
---------------------------------
Biomass - 2025
Max utilization: 0.9979
---------------------------------
Hydro - 2025
Max utilization: 0.9979
---------------------------------
Solar - 2025
Max utilization: 0.7003
---------------------------------
Wind - 2025
Max utilization: 0.8902
---------------------------------


In [13]:
# Check whether any non-solar asset exceeds full-day utilization.
non_solar_df = merge_df[merge_df['energy_source_name'] != 'Solar'].copy()
non_solar_df['utilization'] = non_solar_df['generation_MWh'] / (non_solar_df['generation_capacity_MW'] * 24)

non_solar_df[non_solar_df['utilization'] > 1]

,date_id,asset_id,generation_MWh,company_id,area_id,energy_source_id,commissioning_date_id,asset_name,generation_capacity_MW,zipcode,...,season,is_weekend,is_holiday,is_workday,company_name,parent_company,company_level,energy_source_name,energy_source_group,utilization


In [14]:
# Check whether any solar asset exceeds full-day utilization.
solar_df = merge_df[merge_df['energy_source_name']=='Solar'].copy()

solar_df['utilization'] = solar_df['generation_MWh'] / (solar_df['generation_capacity_MW'] * 16)

solar_df[solar_df['utilization']> 1]

,date_id,asset_id,generation_MWh,company_id,area_id,energy_source_id,commissioning_date_id,asset_name,generation_capacity_MW,zipcode,...,season,is_weekend,is_holiday,is_workday,company_name,parent_company,company_level,energy_source_name,energy_source_group,utilization


In [15]:
# Review whether the generated annual totals by year and energy source meet the defined targets.

final_fact_df.merge(dim_date, on = 'date_id', how = 'left' ).merge(dim_asset, on = 'asset_id', how = 'left').merge(dim_energy_source_h, on='energy_source_id', how='left').groupby(['year', 'energy_source_name'])['generation_MWh'].sum().reset_index()

,year,energy_source_name,generation_MWh
0,2024,Biomass,647000.0
1,2024,Hydro,6400.0
2,2024,Solar,26300.0
3,2024,Wind,1288300.0
4,2025,Biomass,972000.0
5,2025,Hydro,6400.0
6,2025,Solar,54900.0
7,2025,Wind,1271700.0


In [16]:
final_fact_df.to_csv(f"{PROCESSED_DIR}/fact_generation_daily.csv", index=False)
print(f"Exported {len(final_fact_df):,} generation records.")

Exported 428,693 generation records.
